# Project 2 - AI Recommendation Engine (NumPy Only)

The goal here is to build a mini recommendation engine, kind of like a
super simplified version of what Netflix / Spotify do, but only using NumPy.

Each movie already has a feature vector (given to us, made by some other model).
The columns are: Action, Sci-Fi, Romance, Drama, Comedy.

The user also has a vector like this, it just means "how much they like each genre".

Plan (this is basically the todo list from the assignment):
- Phase 1: Similarity Search
- Phase 2: User-to-User Recommendation
- Phase 3: Item-to-Item Recommendation
- Phase 4: Content-Based Recommendation
- Phase 5: Hybrid Recommendation
- Phase 6: Put it all together in one mini system


## Step 0 - Load the data

In [5]:
import numpy as np

np.set_printoptions(suppress=True, precision=2)

movies = np.array([
    [8, 9, 1, 0, 3],
    [2, 1, 9, 8, 7],
    [7, 8, 2, 1, 4],
    [1, 2, 8, 9, 6],
    [9, 9, 0, 1, 2],
    [3, 2, 7, 8, 9],
    [8, 7, 1, 2, 3],
    [2, 3, 9, 7, 8],
    [7, 9, 2, 2, 4],
    [1, 1, 8, 8, 7]
], dtype=float)

movie_names = np.array([
    "The Matrix",
    "Titanic",
    "Inception",
    "Avatar",
    "Interstellar",
    "Joker",
    "The Dark Knight",
    "Forrest Gump",
    "Gladiator",
    "Shutter Island"
])

# columns = Action, Sci-Fi, Romance, Drama, Comedy
feature_names = ["Action", "Sci-Fi", "Romance", "Drama", "Comedy"]

user = np.array([8, 8, 1, 1, 3], dtype=float)

print(movies.shape)
print(user)


(10, 5)
[8. 8. 1. 1. 3.]


## Phase 1 - Similarity Search




In [6]:
def cosine_similarity(a, b):
    # a . b
    dot_product = np.dot(a, b)
    # length (magnitude) of each vector
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

# quick sanity check with itself, should be 1.0
print(cosine_similarity(user, user))


1.0000000000000002


In [7]:
# now let's compute similarity between the user and every single movie
# doing it with a simple loop first because its easier to understand than full vectorizing

sim_scores = []

for i in range(len(movies)):
    score = cosine_similarity(user, movies[i])
    sim_scores.append(score)

sim_scores = np.array(sim_scores)
print(sim_scores)


[0.99 0.37 0.99 0.37 0.99 0.48 0.99 0.47 0.98 0.34]


In [8]:
# now sort to find the best matches. argsort gives ascending order so I flip it
sorted_idx = np.argsort(sim_scores)[::-1]

print("Top movie recommendations based on similarity to user:\n")
for rank, idx in enumerate(sorted_idx, start=1):
    print(f"{rank}. {movie_names[idx]:20s} similarity = {sim_scores[idx]:.4f}")


Top movie recommendations based on similarity to user:

1. The Matrix           similarity = 0.9947
2. The Dark Knight      similarity = 0.9935
3. Interstellar         similarity = 0.9911
4. Inception            similarity = 0.9892
5. Gladiator            similarity = 0.9842
6. Joker                similarity = 0.4834
7. Forrest Gump         similarity = 0.4716
8. Titanic              similarity = 0.3728
9. Avatar               similarity = 0.3669
10. Shutter Island       similarity = 0.3360


## Phase 2 - User-to-User Recommendatio


In [9]:
# made up some other users with different tastes, same 5 features
other_users = np.array([
    [9, 9, 0, 0, 2],   # Alice  - loves action + scifi
    [1, 2, 9, 8, 6],   # Bob    - loves romance + drama
    [5, 5, 5, 5, 5],   # Charlie - likes everything equally
    [2, 2, 8, 9, 7],   # Dana   - romance + drama person
], dtype=float)

other_user_names = ["Alice", "Bob", "Charlie", "Dana"]

user_sim_scores = []
for i in range(len(other_users)):
    score = cosine_similarity(user, other_users[i])
    user_sim_scores.append(score)

user_sim_scores = np.array(user_sim_scores)

for name, score in zip(other_user_names, user_sim_scores):
    print(f"{name:10s} similarity to our user = {score:.4f}")


Alice      similarity to our user = 0.9875
Bob        similarity to our user = 0.3669
Charlie    similarity to our user = 0.7966
Dana       similarity to our user = 0.4177


In [10]:
# find the most similar user
best_user_idx = np.argmax(user_sim_scores)
best_user_name = other_user_names[best_user_idx]
best_user_vector = other_users[best_user_idx]

print(f"Most similar user is: {best_user_name}")

# now recommend movies based on THIS user's taste (user-to-user collaborative filtering idea)
similar_user_scores = []
for i in range(len(movies)):
    score = cosine_similarity(best_user_vector, movies[i])
    similar_user_scores.append(score)

similar_user_scores = np.array(similar_user_scores)
sorted_idx2 = np.argsort(similar_user_scores)[::-1]

print(f"\nBecause you are similar to {best_user_name}, here is what they'd probably like (and so might you):\n")
for rank, idx in enumerate(sorted_idx2, start=1):
    print(f"{rank}. {movie_names[idx]:20s} score = {similar_user_scores[idx]:.4f}")


Most similar user is: Alice

Because you are similar to Alice, here is what they'd probably like (and so might you):

1. Interstellar         score = 0.9970
2. The Matrix           score = 0.9912
3. The Dark Knight      score = 0.9711
4. Inception            score = 0.9588
5. Gladiator            score = 0.9507
6. Joker                score = 0.3399
7. Forrest Gump         score = 0.3291
8. Titanic              score = 0.2256
9. Avatar               score = 0.2219
10. Shutter Island       score = 0.1856


## Phase 3 - Item-to-Item Recommendation

This is the "people who liked X also liked Y" idea (like Amazon's "customers who
bought this also bought").

Step 1: figure out the user's favorite movie so far (just take rank #1 from Phase 1)
Step 2: find movies that are similar to THAT movie, instead of similar to the user


In [11]:
# grabbing the #1 movie from phase 1 as the "seed" movie
seed_idx = sorted_idx[0]
seed_movie_name = movie_names[seed_idx]
seed_movie_vector = movies[seed_idx]

print(f"Seed movie (users favorite so far): {seed_movie_name}")
print(f"Its feature vector: {seed_movie_vector}")


Seed movie (users favorite so far): The Matrix
Its feature vector: [8. 9. 1. 0. 3.]


In [8]:
# now compare this movie to every OTHER movie
item_item_scores = []

for i in range(len(movies)):
    score = cosine_similarity(seed_movie_vector, movies[i])
    item_item_scores.append(score)

item_item_scores = np.array(item_item_scores)

# sort descending, but skip index 0 result which will just be the seed movie itself (sim = 1.0)
sorted_idx3 = np.argsort(item_item_scores)[::-1]

print(f"Because you liked '{seed_movie_name}', you might also like:\n")
count = 0
for idx in sorted_idx3:
    if idx == seed_idx:
        continue  # dont recommend the same movie back to the user
    count += 1
    print(f"{count}. {movie_names[idx]:20s} similarity = {item_item_scores[idx]:.4f}")


Because you liked 'The Matrix', you might also like:

1. Interstellar         similarity = 0.9883
2. Inception            similarity = 0.9853
3. Gladiator            similarity = 0.9774
4. The Dark Knight      similarity = 0.9765
5. Joker                similarity = 0.4243
6. Forrest Gump         similarity = 0.4243
7. Titanic              similarity = 0.3132
8. Avatar               similarity = 0.3063
9. Shutter Island       similarity = 0.2762


## Phase 4 - Content-Based Recommendation



In [9]:
# lets say the user cares A LOT about action/scifi, and barely about the rest
# (just made these numbers up, in a real system this might come from user settings)
feature_weights = np.array([1.5, 1.5, 0.5, 0.5, 0.8])

# apply weights to both user and movies before comparing
weighted_user = user * feature_weights
weighted_movies = movies * feature_weights   # broadcasting the weight vector across all rows

print("weighted user vector:", weighted_user)


weighted user vector: [12.  12.   0.5  0.5  2.4]


In [10]:
content_scores = []
for i in range(len(weighted_movies)):
    score = cosine_similarity(weighted_user, weighted_movies[i])
    content_scores.append(score)

content_scores = np.array(content_scores)
sorted_idx4 = np.argsort(content_scores)[::-1]

print("Content-based recommendations (using feature weights):\n")
for rank, idx in enumerate(sorted_idx4, start=1):
    print(f"{rank}. {movie_names[idx]:20s} score = {content_scores[idx]:.4f}")


Content-based recommendations (using feature weights):

1. Interstellar         score = 0.9980
2. The Matrix           score = 0.9978
3. The Dark Knight      score = 0.9973
4. Inception            score = 0.9957
5. Gladiator            score = 0.9907
6. Forrest Gump         score = 0.6291
7. Joker                score = 0.6190
8. Avatar               score = 0.4843
9. Titanic              score = 0.4706
10. Shutter Island       score = 0.3782


## Phase 5 - Hybrid Recommendation

In [11]:
hybrid_weight_content = 0.6
hybrid_weight_item = 0.4

hybrid_scores = (hybrid_weight_content * content_scores) + (hybrid_weight_item * item_item_scores)

sorted_idx5 = np.argsort(hybrid_scores)[::-1]

print("Hybrid recommendations:\n")
for rank, idx in enumerate(sorted_idx5, start=1):
    print(f"{rank}. {movie_names[idx]:20s} hybrid score = {hybrid_scores[idx]:.4f}")


Hybrid recommendations:

1. The Matrix           hybrid score = 0.9987
2. Interstellar         hybrid score = 0.9941
3. Inception            hybrid score = 0.9915
4. The Dark Knight      hybrid score = 0.9889
5. Gladiator            hybrid score = 0.9854
6. Forrest Gump         hybrid score = 0.5471
7. Joker                hybrid score = 0.5411
8. Avatar               hybrid score = 0.4131
9. Titanic              hybrid score = 0.4076
10. Shutter Island       hybrid score = 0.3374


## Phase 6 - Mini Recommendation System

In [12]:
def recommend_movies(user_vector, movies, movie_names, feature_weights=None, top_n=5):
    '''
    Simple hybrid movie recommender.
    user_vector: the taste vector of the user
    movies: matrix of all movie feature vectors
    movie_names: names lined up with movies rows
    feature_weights: optional, how much to weigh each genre
    top_n: how many recommendations to return
    '''

    if feature_weights is None:
        feature_weights = np.ones(len(user_vector))  # no weighting = all equal

    weighted_user = user_vector * feature_weights
    weighted_movies = movies * feature_weights

    # step 1 - content based scores
    content = np.array([cosine_similarity(weighted_user, m) for m in weighted_movies])

    # step 2 - find favorite movie (highest content score) and do item-item on it
    fav_idx = np.argmax(content)
    fav_vector = movies[fav_idx]
    item_item = np.array([cosine_similarity(fav_vector, m) for m in movies])

    # step 3 - combine
    hybrid = 0.6 * content + 0.4 * item_item

    # step 4 - rank and return top N (skip the exact favorite movie itself since duplicate info)
    ranking = np.argsort(hybrid)[::-1]

    results = []
    for idx in ranking:
        results.append((movie_names[idx], hybrid[idx]))

    return results[:top_n]


In [13]:
# test it with the original user
recs = recommend_movies(user, movies, movie_names, feature_weights=np.array([1.5, 1.5, 0.5, 0.5, 0.8]), top_n=5)

print("Final recommendations for our user:\n")
for name, score in recs:
    print(f"{name:20s} -> {score:.4f}")


Final recommendations for our user:

Interstellar         -> 0.9988
The Matrix           -> 0.9940
The Dark Knight      -> 0.9911
Inception            -> 0.9825
Gladiator            -> 0.9786


In [13]:
# just testing with a totally different user to make sure the function actually works generally
# lets say this new user likes romance + comedy, not action/scifi at all
new_user = np.array([1, 1, 9, 6, 8], dtype=float)

recs2 = recommend_movies(new_user, movies, movie_names, top_n=5)

print("Recommendations for a romance/comedy fan:\n")
for name, score in recs2:
    print(f"{name:20s} -> {score:.4f}")


Recommendations for a romance/comedy fan:

Forrest Gump         -> 0.9919
Titanic              -> 0.9853
Shutter Island       -> 0.9828
Joker                -> 0.9718
Avatar               -> 0.9650
